In [2]:
pip install google-cloud-bigquery google-cloud-storage

   ---------------------------------------- 0.0/5.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.1 MB ? eta -:--:--
   -- ------------------------------------- 0.3/5.1 MB ? eta -:--:--
   -- ------------------------------------- 0.3/5.1 MB ? eta -:--:--
   -- ------------------------------------- 0.3/5.1 MB ? eta -:--:--
   -- ------------------------------------- 0.3/5.1 MB ? eta -:--:--
   -- ------------------------------------- 0.3/5.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.5/5.1 MB 231.1 kB/s eta 0:00:20
   ---- ----------------------------------- 0.5/5.1 MB 231.1 kB/s eta 0:00:20
   ------ --------------------------------- 0.8/5.1 MB 324.8 kB/s eta 0:00:14
   ------ --------------------------------- 0.8/5.1 MB 324.8 kB/s eta 0:00:1


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\sugwo\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [1]:
from google.cloud import bigquery, storage
import pandas as pd

# ── 설정 ──────────────────────────────────────
PROJECT_ID  = 'flight-analysis-2026'
DATASET_ID  = 'flight_analysis'
BUCKET_NAME = 'flight-analysis-2026-bucket'

bq_client  = bigquery.Client(project=PROJECT_ID)
gcs_client = storage.Client(project=PROJECT_ID)
bucket     = gcs_client.bucket(BUCKET_NAME)

print('✅ 클라이언트 연결 완료')

✅ 클라이언트 연결 완료


In [2]:
import os

# ── CSV 파일 경로 ──────────────────────────────
FILES = {
    'flights':        'data/raw/flights.csv',
    'hotels':         'data/raw/hotels.csv',
    'transportation': 'data/raw/transportation.csv',
}

# ── 1단계: GCS 업로드 ─────────────────────────
print('📤 GCS 업로드 중...')
for table_name, file_path in FILES.items():
    blob = bucket.blob(f'raw/{os.path.basename(file_path)}')
    blob.upload_from_filename(file_path)
    print(f'  ✅ {file_path} → gs://{BUCKET_NAME}/raw/{os.path.basename(file_path)}')

# ── 2단계: BigQuery 스키마 정의 ───────────────
schemas = {
    'flights': [
        bigquery.SchemaField('flight_id',         'INTEGER'),
        bigquery.SchemaField('departure_date',     'DATE'),
        bigquery.SchemaField('airline_name',       'STRING'),
        bigquery.SchemaField('airline_type',       'STRING'),
        bigquery.SchemaField('airline_country',    'STRING'),
        bigquery.SchemaField('destination',        'STRING'),
        bigquery.SchemaField('price',              'INTEGER'),
        bigquery.SchemaField('is_weekend',         'BOOLEAN'),
        bigquery.SchemaField('is_peak_season',     'BOOLEAN'),
        bigquery.SchemaField('season_multiplier',  'FLOAT'),
        bigquery.SchemaField('day_of_week',        'STRING'),
        bigquery.SchemaField('month',              'INTEGER'),
    ],
    'hotels': [
        bigquery.SchemaField('hotel_id',              'INTEGER'),
        bigquery.SchemaField('date',                  'DATE'),
        bigquery.SchemaField('destination',           'STRING'),
        bigquery.SchemaField('avg_price_per_night',   'INTEGER'),
        bigquery.SchemaField('is_weekend',            'BOOLEAN'),
        bigquery.SchemaField('is_peak_season',        'BOOLEAN'),
        bigquery.SchemaField('month',                 'INTEGER'),
    ],
    'transportation': [
        bigquery.SchemaField('transport_id',    'INTEGER'),
        bigquery.SchemaField('destination',     'STRING'),
        bigquery.SchemaField('transport_type',  'STRING'),
        bigquery.SchemaField('price',           'INTEGER'),
        bigquery.SchemaField('duration_min',    'INTEGER'),
        bigquery.SchemaField('airport_to_city', 'BOOLEAN'),
    ],
}

# ── 3단계: GCS → BigQuery 로드 ────────────────
print('\n📥 BigQuery 로드 중...')
for table_name, file_path in FILES.items():
    uri        = f'gs://{BUCKET_NAME}/raw/{os.path.basename(file_path)}'
    table_ref  = f'{PROJECT_ID}.{DATASET_ID}.{table_name}'

    job_config = bigquery.LoadJobConfig(
        schema             = schemas[table_name],
        source_format      = bigquery.SourceFormat.CSV,
        skip_leading_rows  = 1,
        write_disposition  = bigquery.WriteDisposition.WRITE_TRUNCATE,
    )

    job = bq_client.load_table_from_uri(uri, table_ref, job_config=job_config)
    job.result()  # 완료까지 대기
    table = bq_client.get_table(table_ref)
    print(f'  ✅ {table_name}: {table.num_rows:,}행 로드 완료')

print('\n 모든 테이블 BigQuery 업로드 완료!')

📤 GCS 업로드 중...
  ✅ data/raw/flights.csv → gs://flight-analysis-2026-bucket/raw/flights.csv
  ✅ data/raw/hotels.csv → gs://flight-analysis-2026-bucket/raw/hotels.csv
  ✅ data/raw/transportation.csv → gs://flight-analysis-2026-bucket/raw/transportation.csv

📥 BigQuery 로드 중...
  ✅ flights: 23,360행 로드 완료
  ✅ hotels: 9,125행 로드 완료
  ✅ transportation: 55행 로드 완료

 모든 테이블 BigQuery 업로드 완료!
